# Notebook 3: The Lakehouse

🎯 **Live notebook** — this runs in Block 4 of the lecture (15 min).

**Pattern:** Open table formats (Iceberg) on object storage with ACID, schema evolution, time travel.

**Stack:** DuckDB + PyIceberg (local Iceberg tables with SQLite catalog)

**Maple Trust Bank** — synthetic BFSI data

---
## Configuration — Plan A / Plan B

Toggle the `USE_PLAN_A` flag below to switch between:
- **Plan A:** watsonx.data + Iceberg (IBM Cloud)
- **Plan B:** Local PyIceberg + DuckDB with SQLite catalog

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
USE_PLAN_A = False  # Set True for watsonx.data + Iceberg; False for local PyIceberg + DuckDB

# Data paths
DATA_DIR = "../data"
LINEAGE_PATH = f"{DATA_DIR}/lineage/lineage_graph.json"

# Local Iceberg catalog config (Plan B)
WAREHOUSE_DIR = "/tmp/iceberg_warehouse"
CATALOG_DB = "/tmp/iceberg_catalog.db"

if USE_PLAN_A:
    print("Plan A: watsonx.data + Iceberg — configure watsonx.data credentials.")
else:
    print("Plan B: Local PyIceberg + DuckDB with SQLite catalog.")

In [ ]:
import duckdb
import pandas as pd
import json
import os
import time
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

---
## Section 1: The Pattern in One Paragraph

The **lakehouse** is what happens when you give a data lake ACID transactions, schema enforcement, and time travel. Open table formats — Apache Iceberg, Delta Lake, Apache Hudi — made this possible by adding a metadata layer on top of object storage files. The key insight: the lakehouse is not a product, it's a *contract* between storage and compute. Any engine that understands Iceberg metadata can read and write the same tables with transactional guarantees. You get the cost and openness of the lake with the reliability of the warehouse. The tradeoff: more operational complexity than a warehouse, and the format wars (Iceberg vs. Delta vs. Hudi) have only recently settled in Iceberg's favor.

---
## Section 2: When You'd Use It, When You Wouldn't

| Use when | Don't use when |
|----------|----------------|
| Unified analytics + ML on open formats | Sub-second OLTP point lookups (use a database) |
| Schema evolution is required (fields added/renamed) | Simple BI on stable schemas (a warehouse is simpler) |
| Time travel / audit trail for regulatory compliance | Your data fits in a single Postgres instance |
| Mixed workloads: BI + data science + streaming | You have no one to operate the format metadata |
| Multi-engine access (Spark, Presto, DuckDB, Flink) | All your data is already in a well-governed warehouse |
| Cost-effective storage with compute elasticity | Your org equates 'lakehouse' with 'no governance needed' |

---
## Section 3: The Setup — Create Local Iceberg Tables

In [ ]:
# Clean up any previous run
for path in [WAREHOUSE_DIR, CATALOG_DB]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

os.makedirs(WAREHOUSE_DIR, exist_ok=True)
print(f"Iceberg warehouse: {WAREHOUSE_DIR}")
print(f"SQLite catalog:    {CATALOG_DB}")

In [ ]:
# Create PyIceberg catalog backed by SQLite
from pyiceberg.catalog.sql import SqlCatalog

catalog = SqlCatalog(
    "maple_trust",
    **{
        "uri": f"sqlite:///{CATALOG_DB}",
        "warehouse": f"file://{WAREHOUSE_DIR}",
    },
)

# Create a namespace for our bank
catalog.create_namespace("bfsi")
print("Created Iceberg catalog 'maple_trust' with namespace 'bfsi'")
print(f"Namespaces: {catalog.list_namespaces()}")

In [ ]:
# Load raw data from parquet
branches_df = pd.read_parquet(f"{DATA_DIR}/branches.parquet")
customers_df = pd.read_parquet(f"{DATA_DIR}/customers.parquet")
accounts_df = pd.read_parquet(f"{DATA_DIR}/accounts.parquet")
transactions_df = pd.read_parquet(f"{DATA_DIR}/transactions.parquet")

print(f"Loaded: branches={len(branches_df)}, customers={len(customers_df)}, "
      f"accounts={len(accounts_df)}, transactions={len(transactions_df)}")

In [ ]:
# Convert DataFrames to PyArrow tables for Iceberg ingestion
branches_arrow = pa.Table.from_pandas(branches_df)
customers_arrow = pa.Table.from_pandas(customers_df)
accounts_arrow = pa.Table.from_pandas(accounts_df)
transactions_arrow = pa.Table.from_pandas(transactions_df)

print("Converted to Arrow tables for Iceberg ingestion")
print(f"  branches schema:     {branches_arrow.schema}")
print(f"  transactions schema: {transactions_arrow.schema}")

In [ ]:
# Create Iceberg tables and load data
print("📊 Reference Architecture Swimlane: Analytical Data Management & Storage")
print("   Sub-zone: 'On Software Hub' — watsonx.data (Iceberg)\n")

tables_config = [
    ("bfsi.branches", branches_arrow),
    ("bfsi.customers", customers_arrow),
    ("bfsi.accounts", accounts_arrow),
    ("bfsi.transactions", transactions_arrow),
]

for table_name, arrow_table in tables_config:
    iceberg_table = catalog.create_table(table_name, schema=arrow_table.schema)
    iceberg_table.append(arrow_table)
    print(f"  Created and loaded: {table_name} ({arrow_table.num_rows:,} rows)")

print(f"\nTables in catalog: {catalog.list_tables('bfsi')}")

In [ ]:
# Inspect the Iceberg metadata structure
txn_table = catalog.load_table("bfsi.transactions")
print("Iceberg metadata for bfsi.transactions:")
print(f"  Location:    {txn_table.location()}")
print(f"  Schema:      {txn_table.schema()}")
print(f"  Snapshots:   {len(txn_table.history())}")
print(f"  Current snapshot: {txn_table.current_snapshot()}")
print()
print("This metadata is what makes the lakehouse different from a lake.")
print("It provides: ACID, time travel, schema tracking, snapshot isolation.")

---
## Section 4: Three Canonical Queries

In [ ]:
# Connect DuckDB and register Iceberg data
# Read Iceberg tables via PyIceberg into Arrow, then register in DuckDB
con = duckdb.connect()

# Scan Iceberg tables to Arrow and register in DuckDB
for tbl_name in ["branches", "customers", "accounts", "transactions"]:
    iceberg_tbl = catalog.load_table(f"bfsi.{tbl_name}")
    arrow_data = iceberg_tbl.scan().to_arrow()
    con.register(tbl_name, arrow_data)

print("Registered Iceberg tables in DuckDB:")
print(con.execute("SHOW TABLES").fetchdf())

### Q1: Total transaction volume by branch for Q3 2024

Query Iceberg tables via DuckDB — works like a warehouse, but on open formats.

In [ ]:
q1 = con.execute("""
    SELECT
        t.branch_id,
        b.name AS branch_name,
        b.region,
        COUNT(*)           AS txn_count,
        SUM(t.amount)      AS total_amount,
        AVG(t.amount)      AS avg_amount
    FROM transactions t
    JOIN branches b ON t.branch_id = b.branch_id
    WHERE t.timestamp >= '2024-07-01' AND t.timestamp < '2024-10-01'
    GROUP BY t.branch_id, b.name, b.region
    ORDER BY total_amount DESC
""").fetchdf()

print("Q1: Transaction volume by branch — Q3 2024 (top 10)")
print("Same query as the lake, same results — but now with ACID guarantees.")
q1.head(10)

### Q2: Find all customers whose policy documents reference AML procedure X

The lakehouse still can't search unstructured documents natively — but it *can* store embeddings alongside structured data.

In [ ]:
print("Q2: Policy document search — AML procedure references")
print("="*60)
print()
print("RESULT: Cannot be answered natively by the lakehouse.")
print()
print("However, the lakehouse has an advantage over the lake:")
print("  → It can store vector embeddings as a column alongside structured data")
print("  → An embedding table in Iceberg could hold doc chunks + vectors")
print("  → A vector-capable engine (watsonx.data + Milvus) could query it")
print()
print("This is the bridge to Notebook 6 (RAG):")
print("  Lakehouse stores the embeddings. RAG pipeline queries them.")
print("  The lakehouse becomes the persistence layer for AI workloads.")

### Q3: Trace the lineage of branch_summary_quarterly back to source

Iceberg metadata provides table-level lineage — snapshots, manifests, history. Better than the lake, but still not full pipeline lineage.

In [ ]:
# Part A: Iceberg snapshot history — table-level lineage
txn_table = catalog.load_table("bfsi.transactions")

print("Q3: Lineage — Iceberg snapshot history for bfsi.transactions")
print("="*60)
print()
for snapshot in txn_table.history():
    print(f"  Snapshot ID:  {snapshot.snapshot_id}")
    print(f"  Timestamp:    {snapshot.timestamp_ms}")
    print()

In [ ]:
# Part B: External lineage graph — pipeline-level lineage
with open(LINEAGE_PATH) as f:
    lineage = json.load(f)

target = "consumed.branch_summary_quarterly"

def trace_lineage(target_node, edges, depth=0):
    upstream = [e for e in edges if e["to"] == target_node]
    for edge in upstream:
        indent = "  " * depth
        print(f"{indent}← {edge['from']}")
        print(f"{indent}   transform: {edge['transform'][:60]}...")
        trace_lineage(edge["from"], edges, depth + 1)

print(f"\nExternal lineage graph for: {target}")
trace_lineage(target, lineage["edges"])
print()
print("✅ Iceberg gives us TABLE-level lineage (snapshots, who wrote what, when).")
print("⚠️  PIPELINE-level lineage (which job transformed what) still needs an external graph.")
print("   → IBM Knowledge Catalog + DataStage provide this in the IBM stack.")

---
## Section 5: Where This Pattern Breaks — Lakehouse Features and Limits

### Feature: Schema Evolution

Add a column, query old and new data seamlessly.

In [ ]:
# Schema evolution: add a 'risk_flag' column to branches
from pyiceberg.types import BooleanType

branches_table = catalog.load_table("bfsi.branches")
print("Before schema evolution:")
print(f"  Columns: {[f.name for f in branches_table.schema().fields]}")

# Add a new column
with branches_table.update_schema() as update:
    update.add_column("risk_flag", BooleanType())

# Reload to see the change
branches_table = catalog.load_table("bfsi.branches")
print("\nAfter schema evolution:")
print(f"  Columns: {[f.name for f in branches_table.schema().fields]}")
print()
print("Old data still reads correctly — the new column is NULL for existing rows.")
print("This is seamless. No ALTER TABLE migration. No downtime.")

# Verify old data reads with new schema
evolved_data = branches_table.scan().to_arrow().to_pandas()
print(f"\nSample after evolution (risk_flag is null for old rows):")
evolved_data[["branch_id", "name", "risk_flag"]].head(5)

### Feature: Time Travel

Query the table as of a previous snapshot.

In [ ]:
# Add more data to create a new snapshot
new_txns = pa.table({
    "transaction_id": [f"TXN-NEW-{i:06d}" for i in range(1000)],
    "account_id": [f"ACCT-{(i % 200000) + 1:06d}" for i in range(1000)],
    "branch_id": [f"MTB-{(i % 50) + 1:03d}" for i in range(1000)],
    "amount": [float(100 + i) for i in range(1000)],
    "currency": ["CAD"] * 1000,
    "timestamp": pd.to_datetime(["2024-12-01"] * 1000),
    "transaction_type": ["deposit"] * 1000,
    "channel": ["online"] * 1000,
    "counterparty_id": [None] * 1000,
})

txn_table = catalog.load_table("bfsi.transactions")
txn_table.append(new_txns)
print("Added 1,000 new transactions (snapshot 2 created)")

In [ ]:
# Time travel: compare snapshots
txn_table = catalog.load_table("bfsi.transactions")
history = txn_table.history()

print("Transaction table snapshot history:")
for i, snapshot in enumerate(history):
    print(f"  Snapshot {i}: ID={snapshot.snapshot_id}, ts={snapshot.timestamp_ms}")

# Current count
current_count = txn_table.scan().to_arrow().num_rows
print(f"\nCurrent snapshot: {current_count:,} rows")

# Previous snapshot (time travel)
if len(history) > 1:
    old_snapshot_id = history[0].snapshot_id
    old_count = txn_table.scan(snapshot_id=old_snapshot_id).to_arrow().num_rows
    print(f"Previous snapshot (ID={old_snapshot_id}): {old_count:,} rows")
    print(f"\nDifference: {current_count - old_count:,} rows added")
    print()
    print("Time travel enables: audit trails, rollback, reproducible analytics.")
    print("Regulators love this — 'show me the data as it was on December 31st.'")

### Feature: Partition Pruning

Iceberg tracks min/max statistics per file, enabling efficient scan pruning.

In [ ]:
# Show the manifest files that Iceberg uses for pruning
txn_table = catalog.load_table("bfsi.transactions")
current_snap = txn_table.current_snapshot()

print("Iceberg manifest information:")
if current_snap and current_snap.manifests(txn_table.io):
    manifests = current_snap.manifests(txn_table.io)
    for i, m in enumerate(manifests[:5]):
        print(f"  Manifest {i}: path={os.path.basename(m.manifest_path)}")
        print(f"    added_rows={m.added_rows_count}, existing_rows={m.existing_rows_count}")

print()
print("Iceberg uses manifest files with column-level min/max stats.")
print("This enables partition pruning — skip entire files that don't match the filter.")
print("Example: a query for Q3 2024 skips all files outside that date range.")

### Break: Sub-millisecond point lookups (OLTP)

The lakehouse is optimized for analytical scans, not single-row seeks.

In [ ]:
# Simulate OLTP-style point lookup: find one specific transaction
target_txn_id = "TXN-NEW-000001"

# Lakehouse scan
start = time.time()
result = txn_table.scan(
    row_filter=f"transaction_id == '{target_txn_id}'"
).to_arrow()
lakehouse_ms = (time.time() - start) * 1000

print(f"Point lookup for transaction_id = '{target_txn_id}'")
print(f"  Lakehouse (Iceberg scan): {lakehouse_ms:.1f} ms — found {result.num_rows} row(s)")
print()
print("For comparison, a database index lookup would be < 1 ms.")
print(f"The lakehouse took {lakehouse_ms:.0f} ms because it scanned manifest files + data files.")
print()
print("⚠️  The lakehouse is optimized for SCAN, not SEEK.")
print("   For OLTP workloads (banking transactions, real-time lookups),")
print("   use a database (Postgres, Db2). The lakehouse is for analytics.")

---
## Section 6: The IBM Stack Mapping

| Component | IBM Product | Swimlane |
|-----------|-------------|----------|
| Lakehouse Engine | watsonx.data (Presto, Spark) | Analytical Data Management & Storage |
| Table Format | Apache Iceberg (native in watsonx.data) | Analytical Data Management & Storage |
| Object Storage | IBM Cloud Object Storage (COS) | Analytical Data Management & Storage |
| Catalog | Iceberg catalog (Hive Metastore or Nessie) | Discovery & Exploration |
| Governance | IBM Knowledge Catalog | Information & Model Management & Governance |

**Swimlane:** The lakehouse sits in the "On Software Hub" box — **watsonx.data IS the lakehouse play for IBM.**

In [ ]:
print("📊 Reference Architecture Swimlane: Analytical Data Management & Storage")
print("   Sub-zone: 'On Software Hub' — watsonx.data")
print()
print("IBM Product Mapping:")
print("  Lakehouse Engine  → watsonx.data (Presto / Spark)")
print("  Table Format      → Apache Iceberg (native support)")
print("  Object Storage    → IBM Cloud Object Storage (COS)")
print("  Catalog           → Iceberg catalog (Hive Metastore or Nessie)")
print("  Governance        → IBM Knowledge Catalog")
print()
print("watsonx.data IS the lakehouse play for IBM.")
print("It bundles Presto + Spark + Iceberg + Milvus into one managed service.")
print("Multi-engine, open format, governed.")

---
## Section 7: BFSI Reality Check

Canadian banks are actively migrating from legacy warehouses (Teradata, Netezza, on-prem Db2) to lakehouse architectures, often on watsonx.data or Databricks. The primary driver is not cost — it's AI. ML and AI workloads need access to the same data that BI dashboards consume, but in open formats that Python and Spark can read natively. Maintaining two copies — one in the warehouse for BI, one in the lake for ML — is expensive, error-prone, and creates governance nightmares (which copy is the source of truth?). The lakehouse eliminates the duplicate by putting everything in Iceberg tables that both Presto (for SQL/BI) and Spark (for ML) can access. The banks that are succeeding started small — one domain, one use case — and proved the model before expanding. The ones that are struggling tried to migrate everything at once. The lakehouse is where the warehouse and the lake finally stop arguing.

In [ ]:
# Clean up
con.close()
print("Notebook 3 complete.")
print()
print("Key takeaway: The lakehouse adds ACID, schema evolution, and time travel")
print("to the data lake — giving you warehouse reliability with lake flexibility.")
print("Next: Notebook 4 (Virtualization) — what if you can't move the data at all?")